# Spark-X2.6-4B-Loop-mHC (`SparkX-Loop v0.2`) — Formal Standalone Inference & Agentic Workspace Suite
### **60 Effective Layers | Sublayer Dual-mHC + 7 3-Tier MoE-Engrams | Runtime $O(n \log n)$ Log-Linear KV Cache | `262K` Token Limit | Formal `SparkXAgenticWorkspace` Runtime**

This inference-only notebook loads the published self-contained formal checkpoint **[`TassP0lar/Spark-X2.6-4B-Loop-mHC`](https://huggingface.co/TassP0lar/Spark-X2.6-4B-Loop-mHC)** directly from Hugging Face Hub and provides:

1. **Cell 1**: Environment Bootstrap & Credentials (`HF_TOKEN`, `OPENCODE_API_KEY`).
2. **Cell 2**: One-Step Formal Checkpoint Loader (`TassP0lar/Spark-X2.6-4B-Loop-mHC`, 60 virtual layers, `dtype=torch.bfloat16`).
3. **Cell 3**: Formal System Prompts + Stateful 60-Layer `_VisitCache` Engine (`DEFAULT_MAX_NEW_TOKENS = 262144` / `262K`) + **`SparkXAgenticWorkspace`** Multi-Turn Coding Runtime (`list_workspace`, `read_file`, `write_file`, `edit_file`, `run_shell`, `verify_webgl_page`).
4. **Cell 4**: **Comprehensive 3D WebGL Test — Voxel Cherry Blossom (Sakura) Garden** inside `SparkXAgenticWorkspace` with 10-point WebGL + Node.js syntax verification and live interactive Colab `iframe` preview.
5. **Cell 5**: Runtime-Only Zero-Collapse $O(n^2) \to O(n \log n)$ Log-Linear NoPE KV Cache benchmark.
6. **Cell 6**: Interactive Multi-Turn Chat & Custom Agentic Task Sandbox (`262K` token budget).


In [ ]:
# === Cell 1: Environment Bootstrap & Credentials (Colab L4 / A100 / G4 GPU) ===
import os, sys, subprocess, shutil
from pathlib import Path

os.environ['HF_HUB_DISABLE_INTERACTIVE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PIP_NO_INPUT'] = '1'

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path.cwd().name == 'Spark-X2.6-Recursion' and (Path.cwd() / 'sparkx_loop_core.py').exists():
    REPO_DIR = Path.cwd()
elif Path('/storage/emulated/10/Documents/Spark_X2.6_Loop/sparkx_loop_core.py').exists():
    REPO_DIR = Path('/storage/emulated/10/Documents/Spark_X2.6_Loop')
else:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=False)
        subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=False)
    os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate', 'safetensors', 'huggingface_hub', 'openai'
], check=True)


def get_secret(*names, default=''):
    for name in names:
        val = os.environ.get(name, '').strip()
        if val:
            return val
    try:
        from google.colab import userdata
        for name in names:
            try:
                val = userdata.get(name)
                if val and str(val).strip():
                    os.environ[name] = str(val).strip()
                    return str(val).strip()
            except Exception:
                pass
    except Exception:
        pass
    return default


HF_TOKEN = get_secret('HF_TOKEN', 'HF_TOKEM')
OPENCODE_API_KEY = get_secret('OPENCODE_API_KEY', 'OPENCODE_API', 'OPENCODE_GO_API_KEY', 'OPENCODE_KEY')

import torch
assert torch.cuda.is_available(), 'CUDA GPU required! Select a GPU runtime in Colab.'
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU Ready: {gpu_name} ({vram_gib:.1f} GiB VRAM)')
print(f'HF_TOKEN loaded: {bool(HF_TOKEN)} | OPENCODE_API_KEY loaded: {bool(OPENCODE_API_KEY)}')


In [ ]:
# === Cell 2: Load Formal Merged Checkpoint (`TassP0lar/Spark-X2.6-4B-Loop-mHC`) ===
import json
from pathlib import Path
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from sparkx_loop_core import (
    LoopSettings, SparkXLoop, attach_lora, merge_lora_in_place
)

FORMAL_REPO_ID = os.environ.get('FORMAL_REPO_ID', 'TassP0lar/Spark-X2.6-4B-Loop-mHC')
FALLBACK_R24_REPO_ID = 'TassP0lar/Spark-X2.6-4B-Loop-mHC-R24'


def load_formal_sparkx_loop(repo_id: str = FORMAL_REPO_ID):
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    try:
        print(f'Downloading formal checkpoint from: {repo_id} ...')
        ckpt_root = Path(snapshot_download(repo_id=repo_id, token=HF_TOKEN or None))
    except Exception as exc:
        print(f'Notice: {repo_id} not reachable ({exc}); falling back to {FALLBACK_R24_REPO_ID} ...')
        ckpt_root = Path(snapshot_download(repo_id=FALLBACK_R24_REPO_ID, token=HF_TOKEN or None))

    ckpt_dir = (ckpt_root / 'stages' / 'R24') if (ckpt_root / 'stages' / 'R24' / 'loop_config.json').exists() else ckpt_root
    meta = json.loads((ckpt_dir / 'loop_config.json').read_text())

    is_formal = bool(meta.get('is_formal_merged_checkpoint', False)) or (ckpt_dir / 'loop_modules.safetensors').exists()
    base_source = str(ckpt_dir) if is_formal else meta.get('base_id', 'XHToken/Spark-X2.6-4B')
    base_rev = None if is_formal else meta.get('base_revision', None)

    print(f'Loading tokenizer & 36-layer backbone ({base_source}) in {dtype} ...')
    tok_inst = AutoTokenizer.from_pretrained(base_source, revision=base_rev, trust_remote_code=True, token=HF_TOKEN or None)
    base_inst = AutoModelForCausalLM.from_pretrained(
        base_source,
        revision=base_rev,
        trust_remote_code=True,
        dtype=dtype,
        device_map={'': 'cuda:0'},
        attn_implementation='eager',
        token=HF_TOKEN or None,
    )
    base_inst.config.use_cache = False

    if (not is_formal) and meta.get('lora_rank', 0) > 0:
        attach_lora(base_inst, start=4, end=36, rank=meta['lora_rank'], alpha=meta['lora_alpha'])

    settings_inst = LoopSettings(
        start=meta.get('loop_start', 4),
        end=meta.get('loop_end', 28),
        passes=meta.get('loop_passes', 2),
        eng_buckets=meta.get('engram_buckets', 16384),
        eng_dim=meta.get('engram_dim', 64),
        eng_experts=meta.get('engram_experts', 16),
        eng_top_k=meta.get('engram_top_k', 2),
        eng_expert_rank=meta.get('eng_expert_rank', meta.get('engram_expert_rank', 16)),
        eng_shared_rank=meta.get('eng_shared_rank', meta.get('engram_shared_rank', 16)),
        backbone_engram_stride=meta.get('backbone_engram_stride', 4),
        checkpoint_layers=False,
        dynamic_mhc=meta.get('dynamic_mhc', True),
        sublayer_mhc=meta.get('sublayer_mhc', True),
        pass_adapter_rank=meta.get('pass_adapter_rank', 16),
    )

    net_inst = SparkXLoop(base_inst, settings_inst, patch_sdpa=True).to('cuda:0')
    weights_file = (ckpt_dir / 'loop_modules.safetensors') if (ckpt_dir / 'loop_modules.safetensors').exists() else (ckpt_dir / 'trainable.safetensors')
    state = load_file(str(weights_file), device='cpu')
    missing, unexpected = net_inst.load_state_dict(state, strict=False)
    del state
    assert not unexpected, f'Unexpected keys: {unexpected[:10]}'

    if not is_formal:
        merge_lora_in_place(net_inst.base)

    net_inst.set_loop(
        meta.get('loop_start', 4),
        meta.get('loop_end', 28),
        meta.get('loop_passes', 2),
        use_mhc=meta.get('mhc_enabled', True),
        use_engram=meta.get('engram_enabled', True),
    )
    net_inst.eval()

    with torch.no_grad():
        ag = [abs(float(torch.tanh(net_inst.controllers[str(i)].extra_gain.float()).item())) for i in range(net_inst.loop_start, net_inst.loop_end)]
        fg = [abs(float(torch.tanh(net_inst.controllers[str(i)].ffn_extra_gain.float()).item())) for i in range(net_inst.loop_start, net_inst.loop_end)]
    print(
        f'Loaded {repo_id} | Virtual Layers = {net_inst.total_virtual_layers} | '
        f'Mean |attn_gate| = {sum(ag)/len(ag):.4f} | Mean |ffn_gate| = {sum(fg)/len(fg):.4f}'
    )
    return net_inst, tok_inst, meta


net, tok, meta = load_formal_sparkx_loop(FORMAL_REPO_ID)


In [ ]:
# === Formal System Prompts + BF16 KV Cache Engine + Anti-Repeat Agentic Watchdog (`SparkXAgenticWorkspace`, 262K Limit) ===
import copy, html, importlib, json, math, os, re, shlex, shutil, subprocess, sys, tempfile, time, types
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import TextStreamer
from transformers.cache_utils import DynamicCache
import sparkx_loop_core
importlib.reload(sparkx_loop_core)
from sparkx_loop_core import (
    _VisitCache,
    patch_spark_attention_for_long_context,
    patch_spark_attention_for_loglinear_kv,
)

DEFAULT_MAX_NEW_TOKENS = 262144  # 262K token limit

# Auto-load formal checkpoint if `net` or `tok` is not yet in memory
if 'net' not in globals() or 'tok' not in globals():
    print('`net` not yet in memory — auto-loading formal checkpoint from Hugging Face (`TassP0lar/Spark-X2.6-4B-Loop-mHC`)...')
    from pathlib import Path as _P
    import json as _json
    import torch as _torch
    from huggingface_hub import snapshot_download as _sd
    from safetensors.torch import load_file as _lf
    from transformers import AutoTokenizer as _AT, AutoModelForCausalLM as _AM
    from sparkx_loop_core import LoopSettings as _LS, SparkXLoop as _SXL, attach_lora as _al, merge_lora_in_place as _mlip

    _repo = os.environ.get('FORMAL_REPO_ID', 'TassP0lar/Spark-X2.6-4B-Loop-mHC')
    _tok_sec = globals().get('HF_TOKEN', os.environ.get('HF_TOKEN', '')) or None
    _dtype = _torch.bfloat16 if _torch.cuda.is_bf16_supported() else _torch.float16
    try:
        _root = _P(_sd(repo_id=_repo, token=_tok_sec))
    except Exception:
        _root = _P(_sd(repo_id='TassP0lar/Spark-X2.6-4B-Loop-mHC-R24', token=_tok_sec))
    _dir = (_root / 'stages' / 'R24') if (_root / 'stages' / 'R24' / 'loop_config.json').exists() else _root
    meta = _json.loads((_dir / 'loop_config.json').read_text())
    _is_formal = bool(meta.get('is_formal_merged_checkpoint', False)) or (_dir / 'loop_modules.safetensors').exists()
    _bsrc = str(_dir) if _is_formal else meta.get('base_id', 'XHToken/Spark-X2.6-4B')
    tok = _AT.from_pretrained(_bsrc, trust_remote_code=True, token=_tok_sec)
    _base = _AM.from_pretrained(_bsrc, trust_remote_code=True, dtype=_dtype, device_map={'': 'cuda:0'}, attn_implementation='eager', token=_tok_sec)
    _base.config.use_cache = False
    if (not _is_formal) and meta.get('lora_rank', 0) > 0:
        _al(_base, start=4, end=36, rank=meta['lora_rank'], alpha=meta['lora_alpha'])
    _settings = _LS(
        start=meta.get('loop_start', 4), end=meta.get('loop_end', 28), passes=meta.get('loop_passes', 2),
        eng_buckets=meta.get('engram_buckets', 16384), eng_dim=meta.get('engram_dim', 64),
        eng_experts=meta.get('engram_experts', 16), eng_top_k=meta.get('engram_top_k', 2),
        eng_expert_rank=meta.get('eng_expert_rank', meta.get('engram_expert_rank', 16)),
        eng_shared_rank=meta.get('eng_shared_rank', meta.get('engram_shared_rank', 16)),
        backbone_engram_stride=meta.get('backbone_engram_stride', 4), checkpoint_layers=False,
        dynamic_mhc=meta.get('dynamic_mhc', True), sublayer_mhc=meta.get('sublayer_mhc', True),
        pass_adapter_rank=meta.get('pass_adapter_rank', 16),
    )
    net = _SXL(_base, _settings, patch_sdpa=True).to('cuda:0')
    _wf = (_dir / 'loop_modules.safetensors') if (_dir / 'loop_modules.safetensors').exists() else (_dir / 'trainable.safetensors')
    net.load_state_dict(_lf(str(_wf), device='cpu'), strict=False)
    if not _is_formal:
        _mlip(net.base)
    net.set_loop(meta.get('loop_start', 4), meta.get('loop_end', 28), meta.get('loop_passes', 2), use_mhc=True, use_engram=True)
    net.eval()


def upgrade_live_net_to_bf16_kv_and_fast_decode(model_obj) -> dict:
    '''Hot-patch an already-loaded SparkXLoop instance in memory so that:
    1. KV Cache (`DynamicCache` + `_VisitCache`) is 100% `torch.bfloat16` (no FP32 RoPE upcast).
    2. `_make_loop_cache_config()` is cached (eliminating `copy.deepcopy` per token).
    3. Single-token decoding (`seq_q == 1`) skips 60-layer 4D mask creation and slices sliding-window layers to 512.
    '''
    from sparkx_loop_core import SparkXLoop as _LatestSparkXLoop
    model_obj._make_loop_cache_config = types.MethodType(_LatestSparkXLoop._make_loop_cache_config, model_obj)
    model_obj._forward_context = types.MethodType(_LatestSparkXLoop._forward_context, model_obj)
    model_obj._apply_layer = types.MethodType(_LatestSparkXLoop._apply_layer, model_obj)
    model_obj.forward_hidden = types.MethodType(_LatestSparkXLoop.forward_hidden, model_obj)
    model_obj._cached_loop_cfg = None

    # Verify KV cache dtype with a 2-token probe
    with torch.no_grad():
        try:
            probe_cache = DynamicCache(config=model_obj._make_loop_cache_config())
        except TypeError:
            probe_cache = DynamicCache()
        probe_ids = torch.tensor([[1, 2]], dtype=torch.long, device='cuda:0')
        _ = model_obj.forward_logits(probe_ids, past_key_values=probe_cache)
        if hasattr(probe_cache, 'layers') and len(probe_cache.layers) > 0:
            k_dt = probe_cache.layers[0].keys.dtype
            v_dt = probe_cache.layers[0].values.dtype
        elif hasattr(probe_cache, 'key_cache') and len(probe_cache.key_cache) > 0:
            k_dt = probe_cache.key_cache[0].dtype
            v_dt = probe_cache.value_cache[0].dtype
        else:
            k_dt = v_dt = model_obj.base.model.embedding.weight.dtype
    return {'key_dtype': str(k_dt), 'value_dtype': str(v_dt), 'virtual_layers': model_obj.total_virtual_layers}


# 1. Formal System Prompt for Direct Chat / Code Synthesis
DIRECT_CHAT_SYSTEM_PROMPT = (
    "You are Spark-X2.6-Loop, a 60-effective-layer Looped Recursive + 3-Tier MoE-Engram AI assistant.\n"
    "Follow these core operating principles:\n"
    "1. Inside `<think>...</think>`, write a concise, structured architectural plan (under 350 words), then close `</think>`.\n"
    "2. Outside `</think>`, deliver the complete, well-formatted, multi-line implementation or answer directly.\n"
    "3. Write clean, readable, multi-line JavaScript/HTML code using standard Three.js APIs (`scene.add(...)`, `requestAnimationFrame(...)`)."
)


# 2. Formal System Prompt Builder for Sandboxed Agentic Workspace Execution
def build_agentic_system_prompt(workspace_root: str, target_file: str = 'index.html') -> str:
    tools_schema = [
        {
            'name': 'list_workspace',
            'description': 'List all files and byte sizes in the active Linux workspace.',
            'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'default': '.'}}},
        },
        {
            'name': 'write_file',
            'description': 'Write complete file contents to disk and automatically run 3D WebGL + JS syntax verification on HTML files.',
            'parameters': {
                'type': 'object',
                'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}},
                'required': ['path', 'content'],
            },
        },
        {
            'name': 'verify_webgl_page',
            'description': 'Run the 10-point 3D WebGL & JavaScript syntax verifier on an HTML file in the workspace.',
            'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'default': target_file}}},
        },
    ]
    schema_json = json.dumps(tools_schema, indent=2)
    ex_call = '{"name": "...", "arguments": {...}}'
    return (
        f"You are Spark-X2.6-Loop, an autonomous software engineering agent operating inside the Linux workspace `{workspace_root}`.\n"
        f"Primary target deliverable: `{workspace_root}/{target_file}`.\n\n"
        f"### Available Workspace Tools\n"
        f"```json\n{schema_json}\n```\n\n"
        f"### Agent Execution Protocol\n"
        f"1. **Concise Deliberation (`<think>...</think>`)**: Outline the 3D scene components in 5-8 brief bullet points, then close `</think>`.\n"
        f"2. **Clean Multi-Line Deliverable**: Outside `</think>`, emit the complete `{target_file}` inside a fenced ```html\n<!DOCTYPE html> ... </html>\n``` block.\n"
        f"   - Write clean, readable multi-line JavaScript (do NOT minify functions onto single lines).\n"
        f"   - Always add meshes to the scene with `scene.add(mesh)` or `group.add(mesh)` and animate with `requestAnimationFrame(animate)`."
    )


def set_loglinear_kv_runtime(
    enabled: bool = True,
    exact_threshold: int = 2048,
    sink_tokens: int = 32,
    local_window: int = 1024,
    base_block: int = 32,
    hip_top_blocks: int = 8,
):
    '''Toggle runtime-only O(n^2) -> O(n log n) BF16 Log-Linear NoPE KV Cache without touching weights.'''
    attn_mod = __import__(type(net.base.model.layers[0].self_attn).__module__, fromlist=['*'])
    attn_mod._sparkx_sdpa_patched = False
    attn_mod._sparkx_loglinear_patched = False
    if enabled:
        n = patch_spark_attention_for_loglinear_kv(
            net.base,
            exact_threshold=exact_threshold,
            sink_tokens=sink_tokens,
            local_window=local_window,
            base_block=base_block,
            hip_top_blocks=hip_top_blocks,
        )
        print(f'Runtime O(n log n) BF16 Log-Linear NoPE KV Cache ENABLED across {n} layers (threshold={exact_threshold}, sinks={sink_tokens}, local={local_window}, HiP_top={hip_top_blocks}x{base_block}).')
    else:
        n = patch_spark_attention_for_long_context(net.base)
        print(f'Standard 1M-Context BF16 Fused SDPA restored across {n} layers.')


def _apply_anti_repeat_penalties(
    logits: torch.Tensor,
    gen_ids: torch.Tensor,
    repetition_penalty: float = 1.02,
    frequency_penalty: float = 0.0,
    presence_penalty: float = 0.0,
    no_repeat_ngram_size: int = 20,
    window: int = 384,
) -> torch.Tensor:
    '''Apply phase-aware anti-repeat penalties + exact sliding N-gram repeat blocker.
    Note: During code synthesis (`in_think=False`), `frequency_penalty` and `presence_penalty` are 0.0
    so repeated JS/HTML tokens (`scene.add`, `new THREE.Mesh`, `;`, `}`) are NEVER syntax-penalized!
    '''
    if gen_ids.numel() == 0:
        return logits
    recent = gen_ids[-window:]

    if repetition_penalty != 1.0 or frequency_penalty > 0.0 or presence_penalty > 0.0:
        uniq, counts = torch.unique(recent, return_counts=True)
        if repetition_penalty != 1.0:
            score = logits[0, uniq]
            logits[0, uniq] = torch.where(score < 0, score * repetition_penalty, score / repetition_penalty)
        if frequency_penalty > 0.0 or presence_penalty > 0.0:
            logits[0, uniq] -= (counts.to(logits.dtype) * frequency_penalty + presence_penalty)

    n = int(no_repeat_ngram_size)
    seq_len = recent.numel()
    if n >= 4 and seq_len >= n:
        prefix = recent[-(n - 1):]
        windows = recent[:-1].unfold(0, n - 1, 1)
        matches = (windows == prefix.unsqueeze(0)).all(dim=-1)
        if matches.any():
            match_indices = matches.nonzero(as_tuple=False).squeeze(-1) + (n - 1)
            banned_tokens = recent[match_indices]
            logits[0, banned_tokens] = -1e9
    return logits


def _sample_next_token(
    logits: torch.Tensor,
    gen_ids: torch.Tensor,
    temperature: float = 0.6,
    top_p: float = 0.92,
    top_k: int = 40,
    repetition_penalty: float = 1.02,
    frequency_penalty: float = 0.0,
    presence_penalty: float = 0.0,
    no_repeat_ngram_size: int = 24,
    banned_token_ids: list[int] | None = None,
) -> torch.Tensor:
    logits = logits.clone()
    if banned_token_ids:
        for bid in banned_token_ids:
            if isinstance(bid, int) and 0 <= bid < logits.shape[-1]:
                logits[0, bid] = -1e9
    logits = _apply_anti_repeat_penalties(
        logits,
        gen_ids,
        repetition_penalty=repetition_penalty,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
    )
    if temperature <= 1e-5:
        return logits.argmax(dim=-1, keepdim=True)
    scaled = logits / float(temperature)
    if top_k > 0 and top_k < scaled.shape[-1]:
        topk_vals, _ = torch.topk(scaled, k=top_k, dim=-1)
        scaled = torch.where(scaled < topk_vals[:, -1:], torch.full_like(scaled, -1e9), scaled)
    probs = F.softmax(scaled, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = (cum_probs - sorted_probs) > top_p
    sorted_probs[mask] = 0.0
    sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True).clamp_min(1e-12)
    sampled = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx.gather(-1, sampled)


def _has_phrase_loop(token_list: list[int], min_pat: int = 8, max_pat: int = 48) -> bool:
    '''Detect if the tail of token_list contains a repeating phrase of length [min_pat..max_pat].'''
    n = len(token_list)
    if n < min_pat * 2:
        return False
    for pat_len in range(min_pat, min(max_pat, n // 2) + 1):
        if token_list[-pat_len:] == token_list[-2 * pat_len : -pat_len]:
            return True
    return False


@torch.no_grad()
def generate_sparkx_loop(
    messages=None,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    max_think_tokens: int = 900,
    enable_thinking: bool = True,
    reasoning_effort: str = 'high',
    temperature: float = 0.6,
    top_p: float = 0.92,
    top_k: int = 40,
    repetition_penalty: float = 1.04,
    frequency_penalty: float = 0.22,
    presence_penalty: float = 0.12,
    stream: bool = True,
    system_prompt: str | None = None,
    post_think_anchor: str = '',
    stop_on_html_close: bool = True,
):
    '''Stateful 60-layer BF16 KV-cached generation with Phase-Aware Anti-Repeat & Spurious </think> Shield.'''
    net.eval()
    msg_list = list(messages or [])
    if not any(m.get('role') == 'system' for m in msg_list):
        sys_content = system_prompt if system_prompt is not None else DIRECT_CHAT_SYSTEM_PROMPT
        msg_list = [{'role': 'system', 'content': sys_content}] + msg_list

    prompt_text = tok.apply_chat_template(
        msg_list,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
        reasoning_effort=reasoning_effort,
    )
    if enable_thinking and not prompt_text.rstrip().endswith('<think>'):
        prompt_text = prompt_text + '<think>\n'
    elif (not enable_thinking) and post_think_anchor:
        prompt_text = prompt_text + post_think_anchor

    enc = tok(prompt_text, return_tensors='pt', add_special_tokens=False)
    out = enc['input_ids'].to('cuda:0')
    prompt_len = out.shape[-1]

    try:
        cache = DynamicCache(config=net._make_loop_cache_config())
    except TypeError:
        cache = DynamicCache()

    if stream:
        if enable_thinking:
            sys.stdout.write('<think>\n')
        elif post_think_anchor:
            sys.stdout.write(post_think_anchor)
        sys.stdout.flush()

    in_think = bool(enable_thinking)
    anchor_injected = bool((not enable_thinking) and post_think_anchor)
    html_started = bool('<!doctype html' in prompt_text.lower() or '<html' in prompt_text.lower())

    eos_ids = {tok.eos_token_id}
    for extra_eos in ('<|im_end|>', '<|endoftext|>', '<｜end▁of▁sentence｜>'):
        tid = tok.convert_tokens_to_ids(extra_eos)
        if isinstance(tid, int) and tid >= 0:
            eos_ids.add(tid)

    think_start_id = tok.convert_tokens_to_ids('<think>')
    think_end_id = tok.convert_tokens_to_ids('</think>')
    think_special_ids = [t for t in (think_start_id, think_end_id) if isinstance(t, int) and t >= 0]

    streamer = TextStreamer(tok, skip_prompt=False, skip_special_tokens=False) if stream else None
    t0 = time.perf_counter()

    # Prefill full prompt into BF16 KV cache (`out` and `cache` have exact length `prompt_len`)
    logits = net.forward_logits(out, past_key_values=cache)[:, -1, :].float()

    for step_idx in range(max_new_tokens):
        gen_so_far_ids = out[0, prompt_len:]
        n_gen_so_far = int(gen_so_far_ids.numel())

        # Build dynamic banned token list:
        # 1. Once outside <think>, NEVER allow spurious <think> or </think> tokens (fixes v0.2 mid-code </think> spike)!
        # 2. While an HTML document is open and </html> hasn't closed yet (up to 12K tokens), suppress premature EOS!
        banned_ids = []
        if not in_think:
            banned_ids.extend(think_special_ids)
            if html_started and stop_on_html_close and n_gen_so_far < 12288:
                banned_ids.extend(list(eos_ids))

        # Phase-aware sampling parameters:
        # - In <think>: apply count-based frequency/presence penalties to prevent prose loops.
        # - Outside </think> (code): 0.0 frequency/presence penalty so JS/Three.js syntax is never corrupted!
        cur_temp = temperature if in_think else max(0.35, temperature * 0.85)
        cur_rep = repetition_penalty if in_think else 1.01
        cur_freq = frequency_penalty if in_think else 0.0
        cur_pres = presence_penalty if in_think else 0.0
        cur_ngram = 10 if in_think else 28

        next_id = _sample_next_token(
            logits,
            gen_so_far_ids,
            temperature=cur_temp,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=cur_rep,
            frequency_penalty=cur_freq,
            presence_penalty=cur_pres,
            no_repeat_ngram_size=cur_ngram,
            banned_token_ids=banned_ids,
        )

        tok_id = int(next_id.item())
        if streamer is not None:
            streamer.put(next_id.cpu())
        prev_tok = out[:, -2:]
        out = torch.cat([out, next_id], dim=-1)

        if tok_id in eos_ids:
            break

        gen_so_far_ids = out[0, prompt_len:]
        n_gen_so_far = int(gen_so_far_ids.numel())

        # Advance BF16 KV cache by feeding `next_id` so `cache` and `out` are always 100% synchronized!
        logits = net.forward_logits(next_id, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()

        # Check natural </think> transition
        if in_think and (
            (isinstance(think_end_id, int) and tok_id == think_end_id)
            or (n_gen_so_far % 4 == 0 and '</think>' in tok.decode(gen_so_far_ids[-8:], skip_special_tokens=False))
        ):
            in_think = False
            if post_think_anchor and not anchor_injected:
                anchor_injected = True
                if streamer is not None:
                    streamer.end()
                anc_ids = tok.encode(post_think_anchor, add_special_tokens=False, return_tensors='pt').to('cuda:0')
                if stream:
                    sys.stdout.write(post_think_anchor)
                    sys.stdout.flush()
                prev_tok = out[:, -2:]
                logits = net.forward_logits(anc_ids, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
                out = torch.cat([out, anc_ids], dim=-1)
                if '<!doctype html' in post_think_anchor.lower() or '<html' in post_think_anchor.lower():
                    html_started = True
                continue

        # Agentic Watchdog Rule 1: If <think> exceeds max_think_tokens or enters a phrase loop,
        # cleanly close </think> and inject post_think_anchor in exact lockstep with the KV cache!
        if in_think and (
            n_gen_so_far >= max_think_tokens
            or (n_gen_so_far >= 64 and n_gen_so_far % 16 == 0 and _has_phrase_loop(gen_so_far_ids[-160:].tolist()))
        ):
            in_think = False
            anchor_injected = True
            if streamer is not None:
                streamer.end()
            bridge = '.\n</think>\n' + (post_think_anchor or '')
            bridge_ids = tok.encode(bridge, add_special_tokens=False, return_tensors='pt').to('cuda:0')
            if stream:
                sys.stdout.write(bridge)
                sys.stdout.flush()
            prev_tok = out[:, -2:]
            logits = net.forward_logits(bridge_ids, past_key_values=cache, prev_ids=prev_tok)[:, -1, :].float()
            out = torch.cat([out, bridge_ids], dim=-1)
            if '<!doctype html' in bridge.lower() or '<html' in bridge.lower():
                html_started = True
            continue

        # Agentic Watchdog Rule 2: Track HTML opening and stop immediately when </html> or </tool_call> is completed
        if (not in_think) and (n_gen_so_far % 4 == 0):
            tail_txt = tok.decode(gen_so_far_ids[-24:], skip_special_tokens=False)
            low_tail = tail_txt.lower()
            if '<!doctype html' in low_tail or '<html' in low_tail:
                html_started = True
            if '</tool_call>' in tail_txt:
                break
            if stop_on_html_close and '</html>' in low_tail:
                break

    if streamer is not None:
        streamer.end()

    elapsed = time.perf_counter() - t0
    gen_ids = out[0, prompt_len:]
    raw_text = tok.decode(gen_ids, skip_special_tokens=False)
    if enable_thinking and '</think>' in raw_text and not raw_text.lstrip().startswith('<think>'):
        raw_text = '<think>\n' + raw_text
    think_text, final_answer = '', raw_text
    if '<think>' in raw_text and '</think>' in raw_text:
        think_text = raw_text.split('<think>', 1)[1].split('</think>', 1)[0].strip()
        final_answer = raw_text.split('</think>', 1)[1]
    elif (not enable_thinking) and post_think_anchor:
        final_answer = post_think_anchor + raw_text
    for eos_str in ('<|im_end|>', '<|endoftext|>', '<｜end▁of▁sentence｜>'):
        final_answer = final_answer.replace(eos_str, '')
    final_answer = final_answer.strip()

    n_gen = int(gen_ids.numel())
    return {
        'raw_text': raw_text,
        'think': think_text,
        'answer': final_answer,
        'tokens': n_gen,
        'tok_per_s': n_gen / max(elapsed, 1e-6),
        'elapsed_s': elapsed,
        'cache': cache,
        'full_ids': out,
    }


class SparkXAgenticWorkspace:
    '''Formal Sandboxed Multi-Turn Agentic Workspace Environment for Spark-X2.6-Loop.

    Combines real filesystem tools (`list_workspace`, `read_file`, `write_file`, `edit_file`),
    sandboxed shell execution (`run_shell`), automated 10-point 3D WebGL + Node.js syntax verification
    (`verify_webgl_page`), and Automated Agentic Repair!
    '''
    def __init__(self, workspace_dir: str = '/content/sakura_agent_workspace'):
        self.root = Path(workspace_dir).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.tool_log: list[dict] = []

    def _safe_path(self, rel_path: str) -> Path:
        clean = str(rel_path).replace('\\', '/').split('/')[-1] if (':' in str(rel_path) or str(rel_path).startswith('/')) else str(rel_path)
        p = (self.root / clean).resolve()
        if not str(p).startswith(str(self.root)):
            p = self.root / Path(clean).name
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def list_workspace(self, path: str = '.') -> dict:
        target = self._safe_path(path) if path != '.' else self.root
        files = []
        for item in sorted(target.rglob('*')):
            if item.is_file():
                files.append({
                    'path': str(item.relative_to(self.root)),
                    'bytes': item.stat().st_size,
                })
        res = {
            'tool': 'list_workspace',
            'workspace_root': str(self.root),
            'files': files,
            'three_js_cdn': 'https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js',
            'status': 'Workspace verified and ready. Write the complete <!DOCTYPE html>...</html> implementation to index.html now.',
        }
        self.tool_log.append(res)
        return res

    def read_file(self, path: str) -> dict:
        p = self._safe_path(path)
        if not p.exists():
            res = {'tool': 'read_file', 'path': p.name, 'error': 'File does not exist yet.'}
        else:
            content = p.read_text(encoding='utf-8', errors='replace')
            res = {'tool': 'read_file', 'path': p.name, 'bytes': len(content), 'preview': content[:1500]}
        self.tool_log.append(res)
        return res

    def write_file(self, path: str, content: str) -> dict:
        p = self._safe_path(path)
        p.write_text(content, encoding='utf-8')
        res = {
            'tool': 'write_file',
            'path': str(p.relative_to(self.root)),
            'absolute_path': str(p),
            'bytes_written': len(content.encode('utf-8')),
            'lines_written': content.count('\n') + 1,
            'status': 'success',
        }
        if p.suffix.lower() in ('.html', '.htm'):
            res['verification'] = self.verify_webgl_page(p.name)
        self.tool_log.append(res)
        return res

    def edit_file(self, path: str, old_text: str, new_text: str) -> dict:
        p = self._safe_path(path)
        if not p.exists():
            return {'tool': 'edit_file', 'path': p.name, 'error': 'File not found.'}
        txt = p.read_text(encoding='utf-8')
        if old_text not in txt:
            return {'tool': 'edit_file', 'path': p.name, 'error': 'old_text not found in file.'}
        updated = txt.replace(old_text, new_text, 1)
        p.write_text(updated, encoding='utf-8')
        res = {'tool': 'edit_file', 'path': p.name, 'bytes_written': len(updated), 'status': 'success'}
        if p.suffix.lower() in ('.html', '.htm'):
            res['verification'] = self.verify_webgl_page(p.name)
        self.tool_log.append(res)
        return res

    def run_shell(self, command: str) -> dict:
        allowed_prefixes = ('ls', 'wc', 'head', 'tail', 'grep', 'cat', 'node', 'python3', 'stat')
        parts = shlex.split(command) if command.strip() else []
        if not parts or parts[0] not in allowed_prefixes:
            return {'tool': 'run_shell', 'command': command, 'error': f'Command must start with one of {allowed_prefixes}'}
        try:
            proc = subprocess.run(
                parts, cwd=str(self.root), capture_output=True, text=True, timeout=10
            )
            res = {
                'tool': 'run_shell',
                'command': command,
                'returncode': proc.returncode,
                'stdout': proc.stdout[:1500],
                'stderr': proc.stderr[:1500],
            }
        except Exception as exc:
            res = {'tool': 'run_shell', 'command': command, 'error': str(exc)}
        self.tool_log.append(res)
        return res

    def verify_webgl_page(self, path: str = 'index.html') -> dict:
        p = self._safe_path(path)
        if not p.exists():
            return {'passed': 0, 'total': 10, 'ok': False, 'error': f'{p.name} does not exist'}
        html_code = p.read_text(encoding='utf-8', errors='replace')
        checks = {
            '1. Valid HTML5 Document (<!DOCTYPE html> ... </html>)': bool(re.search(r'<!DOCTYPE html', html_code, re.I) and re.search(r'</html>', html_code, re.I)),
            '2. Three.js Library Included': ('three' in html_code.lower() and 'THREE' in html_code),
            '3. WebGLRenderer & PerspectiveCamera': ('WebGLRenderer' in html_code and 'PerspectiveCamera' in html_code),
            '4. Voxel BoxGeometry / InstancedMesh': ('BoxGeometry' in html_code or 'InstancedMesh' in html_code),
            '5. Sakura Blossom Palette (Pink/Blossom Hex Colors)': bool(re.search(r'ff[89a-f][0-9a-f]{3}|ffb7c5|ff8da1|ffd1dc|ffc0cb|sakura|blossom', html_code, re.I)),
            '6. Pond / Bridge / Torii / Lantern Voxel Structures': bool(re.search(r'pond|water|bridge|torii|lantern', html_code, re.I)),
            '7. Dynamic Lighting (Ambient/Directional/PointLight)': ('Light' in html_code),
            '8. Falling Petal Animation Loop (requestAnimationFrame)': bool('requestAnimationFrame' in html_code or 'setInterval' in html_code),
            '9. Interactive Camera / Mouse / UI Controls': bool(re.search(r'mousedown|pointerdown|wheel|addEventListener|onclick', html_code, re.I)),
            '10. Window Resize Handler': ('resize' in html_code.lower()),
        }
        js_syntax_ok = True
        js_syntax_msg = 'not_checked'
        inline_scripts = [
            m.group(1) for m in re.finditer(r'<script(?![^>]*\bsrc=)[^>]*>([\s\S]*?)</script>', html_code, re.I)
            if m.group(1).strip()
        ]
        if inline_scripts and shutil.which('node'):
            js_file = self.root / '_syntax_check.js'
            js_file.write_text('\n'.join(inline_scripts), encoding='utf-8')
            proc = subprocess.run(['node', '--check', str(js_file)], capture_output=True, text=True)
            js_file.unlink(missing_ok=True)
            js_syntax_ok = (proc.returncode == 0)
            js_syntax_msg = 'OK' if js_syntax_ok else proc.stderr.strip()[:600]

        passed = sum(1 for v in checks.values() if v)
        missing = [k for k, v in checks.items() if not v]
        return {
            'file': p.name,
            'bytes': len(html_code),
            'passed': passed,
            'total': len(checks),
            'js_syntax_ok': js_syntax_ok,
            'js_syntax_msg': js_syntax_msg,
            'missing_checks': missing,
            'checks': checks,
            'ok': (passed >= 8 and checks['1. Valid HTML5 Document (<!DOCTYPE html> ... </html>)'] and js_syntax_ok),
        }

    @staticmethod
    def extract_html(text: str) -> str:
        m = re.search(r'```(?:html|HTML)?\s*(<!DOCTYPE html[\s\S]*?</html>)\s*```', text, re.I)
        if m:
            return m.group(1).strip()
        m2 = re.search(r'(<!DOCTYPE html[\s\S]*?</html>|<html[\s\S]*?</html>)', text, re.I)
        if m2:
            return m2.group(1).strip()
        return ''

    def parse_and_execute_turn(self, assistant_text: str, raw_text: str, target_file: str = 'index.html') -> tuple[list[dict], bool]:
        '''Detect and execute any explicit <tool_call>, XML tool, code block, or implicit workspace-check intent.'''
        results = []

        html_doc = self.extract_html(assistant_text) or self.extract_html(raw_text)
        if html_doc:
            w_res = self.write_file(target_file, html_doc)
            results.append(w_res)
            is_done = bool(w_res.get('verification', {}).get('ok', False))
            return results, is_done

        for tc_match in re.finditer(r'<tool_call>\s*(\{[\s\S]*?\})\s*</tool_call>', assistant_text):
            try:
                payload = json.loads(tc_match.group(1))
                name = str(payload.get('name') or payload.get('tool') or '').lower()
                args = payload.get('arguments') or payload.get('parameters') or {}
                if isinstance(args, str):
                    args = json.loads(args)
                if name in ('list_workspace', 'ls', 'list_dir', 'check_workspace'):
                    results.append(self.list_workspace(args.get('path', '.')))
                elif name in ('read_file', 'read', 'cat'):
                    results.append(self.read_file(args.get('path', target_file)))
                elif name in ('write_file', 'write', 'create_file'):
                    w_res = self.write_file(args.get('path', target_file), args.get('content', ''))
                    results.append(w_res)
                    if w_res.get('verification', {}).get('ok', False):
                        return results, True
                elif name in ('edit_file', 'edit', 'replace'):
                    e_res = self.edit_file(args.get('path', target_file), args.get('old_text', ''), args.get('new_text', ''))
                    results.append(e_res)
                    if e_res.get('verification', {}).get('ok', False):
                        return results, True
                elif name in ('run_shell', 'bash', 'shell'):
                    results.append(self.run_shell(args.get('command', 'ls -la')))
                elif name in ('verify_webgl_page', 'verify'):
                    v_res = self.verify_webgl_page(args.get('path', target_file))
                    results.append(v_res)
                    if v_res.get('ok', False):
                        return results, True
            except Exception as exc:
                results.append({'tool': 'tool_call_parse_error', 'error': str(exc)})

        if results:
            return results, False

        ws_res = self.list_workspace('.')
        results.append(ws_res)
        return results, False

    def run_agentic_task(
        self,
        task_prompt: str,
        target_file: str = 'index.html',
        max_turns: int = 3,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        max_think_tokens: int = 900,
        temperature: float = 0.6,
        seed_turn1_result: dict | None = None,
    ) -> dict:
        '''Execute a multi-turn agentic coding session using native Spark-X2.6 chat templates & BF16 KV cache.'''
        (self.root / 'README_SPEC.md').write_text(task_prompt, encoding='utf-8')
        initial_ws = self.list_workspace('.')
        agent_sys = build_agentic_system_prompt(str(self.root), target_file=target_file)
        enriched_user_prompt = (
            f"{task_prompt}\n\n"
            f"<tool_response>\n{json.dumps(initial_ws, ensure_ascii=False)}\n</tool_response>\n"
            f"The workspace `{self.root}` has already been inspected above. "
            f"Plan the 3D scene concisely inside `<think>...</think>`, and after `</think>` immediately output the complete, multi-line `{target_file}` inside ```html\n<!DOCTYPE html>...</html>\n```."
        )
        messages = [
            {'role': 'system', 'content': agent_sys},
            {'role': 'user', 'content': enriched_user_prompt},
        ]

        html_anchor = f"Here is the complete `{target_file}` implementation for `{self.root}/{target_file}`:\n\n```html\n<!DOCTYPE html>\n<html lang=\"en\">\n"
        turns_trace = []

        for turn_idx in range(1, max_turns + 1):
            print('\n' + '=' * 98)
            print(f'[AGENTIC WORKSPACE TURN {turn_idx}/{max_turns}] Workspace: {self.root} | temp={temperature} | KV=BF16')
            print('=' * 98)

            if turn_idx == 1 and seed_turn1_result is not None and seed_turn1_result.get('raw_text'):
                print('[Reusing existing Turn 1 output already in notebook memory...]')
                res = seed_turn1_result
            else:
                res = generate_sparkx_loop(
                    messages,
                    max_new_tokens=max_new_tokens,
                    max_think_tokens=max_think_tokens,
                    enable_thinking=(turn_idx == 1),
                    reasoning_effort='high',
                    temperature=temperature,
                    stream=True,
                    system_prompt=agent_sys,
                    post_think_anchor=html_anchor,
                    stop_on_html_close=True,
                )

            ans_txt = res.get('answer', '')
            raw_txt = res.get('raw_text', '')
            if '</html>' in ans_txt.lower() and '<!doctype html' not in ans_txt.lower():
                ans_txt = '<!DOCTYPE html>\n<html lang="en">\n' + ans_txt

            tool_outputs, is_done = self.parse_and_execute_turn(ans_txt, raw_txt, target_file=target_file)
            turns_trace.append({
                'turn': turn_idx,
                'tokens': res.get('tokens', 0),
                'tok_per_s': round(res.get('tok_per_s', 0.0), 2),
                'elapsed_s': round(res.get('elapsed_s', 0.0), 2),
                'tool_outputs': tool_outputs,
                'done': is_done,
            })

            print(f"\n[Turn {turn_idx} Speed]: {res.get('tokens', 0)} tokens in {res.get('elapsed_s', 0.0):.1f}s ({res.get('tok_per_s', 0.0):.1f} tok/s)")
            print(f'[Agentic Harness — Turn {turn_idx} Tool Execution]:')
            print(json.dumps(tool_outputs, indent=2))

            if is_done:
                print(f'\n[Agentic Task Complete on Turn {turn_idx}!] Verified file: {self.root / target_file}')
                break

            # Build clean native Spark-X2.6 multi-turn messages for next turn (using tok.apply_chat_template!)
            ver = tool_outputs[-1].get('verification', self.verify_webgl_page(target_file))
            err_msg = ver.get('js_syntax_msg', '')
            missing_list = ver.get('missing_checks', [])
            messages = [
                {'role': 'system', 'content': agent_sys},
                {
                    'role': 'user',
                    'content': (
                        f"{task_prompt}\n\n"
                        f"<tool_response>\n{json.dumps({'verification': {'passed': ver.get('passed'), 'js_syntax_ok': ver.get('js_syntax_ok'), 'js_syntax_msg': err_msg, 'missing_checks': missing_list}}, ensure_ascii=False)}\n</tool_response>\n"
                        f"Write a clean, multi-line `{target_file}` with valid JavaScript syntax (`scene.add(mesh)`, `requestAnimationFrame(animate)`), fixing: {err_msg or missing_list}."
                    ),
                },
            ]

        final_file = self.root / target_file
        final_html = final_file.read_text(encoding='utf-8') if final_file.exists() else ''
        if final_html and not final_html.lstrip().lower().startswith('<!doctype html'):
            final_html = '<!DOCTYPE html>\n<html lang="en">\n' + final_html
            final_file.write_text(final_html, encoding='utf-8')

        final_audit = self.verify_webgl_page(target_file) if final_file.exists() else {'passed': 0, 'total': 10, 'ok': False, 'checks': {}}
        return {
            'workspace': str(self.root),
            'target_file': str(final_file),
            'html_code': final_html,
            'verification': final_audit,
            'turns': turns_trace,
        }


kv_status = upgrade_live_net_to_bf16_kv_and_fast_decode(net)
set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)
print(f"BF16 KV Cache Verified: {kv_status} | Default Temp = 0.6 | Limit = {DEFAULT_MAX_NEW_TOKENS} tokens (262K).")


In [ ]:
# === Formal Agentic 3D WebGL Build & Verification — Voxel Cherry Blossom (Sakura) Garden ===
import html
from pathlib import Path
from IPython.display import display, HTML

SAKURA_3D_PROMPT = (
    "Build a complete, self-contained, single-file 3D interactive web page (`index.html`, `<!DOCTYPE html> ... </html>`) "
    "that renders a **Voxel Cherry Blossom (Sakura) Garden** using Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`).\n\n"
    "Your implementation MUST include all of the following visual & interactive features:\n"
    "1. **Procedural Voxel Island & Pond**: A multi-layered voxel garden island (grass, soil, stone border voxels) with a central tranquil reflective water pond and voxel stepping stones.\n"
    "2. **Traditional Red Voxel Torii Gate or Arched Wooden Bridge**: Spanning across or beside the pond.\n"
    "3. **Procedural Voxel Cherry Blossom (Sakura) Trees**: Multiple gnarled dark-wood voxel trunks and branches crowned with lush multi-tone pink (`#ffb7c5`, `#ff8da1`, `#ffd1dc`, `#fff0f5`) voxel blossom canopies.\n"
    "4. **Japanese Stone Lanterns (`Toro`) with Warm Glowing PointLights**: Placed along the garden path casting warm golden light.\n"
    "5. **Animated Falling Sakura Petal Particle System**: Hundreds of pink voxel petals drifting, swaying in a sinusoidal breeze, rotating in 3D, and respawning when they touch the water/ground (`requestAnimationFrame`).\n"
    "6. **Interactive Camera & Lighting HUD**: Custom mouse-drag orbit + scroll-wheel zoom camera controller, a gentle auto-rotate toggle, and a UI button to cycle between **Spring Daylight**, **Golden Sunset**, and **Lantern Night** atmospheres.\n\n"
    "Write clean, readable multi-line JavaScript inside `<script>...</script>` (do not minify onto single lines)."
)

agent_workspace = SparkXAgenticWorkspace('/content/sakura_agent_workspace')

agent_result = agent_workspace.run_agentic_task(
    SAKURA_3D_PROMPT,
    target_file='index.html',
    max_turns=3,
    max_new_tokens=DEFAULT_MAX_NEW_TOKENS,  # 262,144 (262K)
    max_think_tokens=900,
    temperature=0.6,
    seed_turn1_result=None,
)

ver = agent_result['verification']
print('\n' + '=' * 98)
print(f"FINAL AGENTIC WORKSPACE AUDIT ({ver['passed']}/{ver['total']} checks passed | JS Syntax: {ver.get('js_syntax_msg', 'N/A')})")
print('=' * 98)
for check_name, ok_flag in ver.get('checks', {}).items():
    print(f"  [{'PASS' if ok_flag else 'MISS'}] {check_name}")
print(f"\nSaved Deliverable: {agent_result['target_file']} ({len(agent_result['html_code']):,} bytes across {len(agent_result['turns'])} agentic turn(s))")

if agent_result['html_code']:
    escaped_html = html.escape(agent_result['html_code'], quote=True)
    iframe_html = f'''
    <div style="margin-top:12px; border:2px solid #ffb7c5; border-radius:12px; overflow:hidden; box-shadow:0 8px 28px rgba(0,0,0,0.35);">
      <div style="background:#1a1625; color:#ffd1dc; padding:8px 14px; font-family:monospace; font-size:13px; display:flex; justify-content:space-between;">
        <span>Spark-X2.6-4B-Loop-mHC (60L Agentic Workspace) — Live 3D Voxel Cherry Blossom Garden</span>
        <span>{len(agent_result['html_code']):,} bytes | {ver['passed']}/{ver['total']} Checks | Drag to Orbit / Scroll to Zoom</span>
      </div>
      <iframe srcdoc="{escaped_html}" style="width:100%; height:640px; border:none; background:#0d0b14;"></iframe>
    </div>
    '''
    display(HTML(iframe_html))


In [ ]:
# === Cell 7: Runtime-Only O(n^2) -> O(n log n) Log-Linear NoPE KV Cache Verification Benchmark ===
import time
import torch

print('=== Testing Runtime-Only O(n log n) Log-Linear NoPE KV Cache (Zero Checkpoint Weight Changes) ===')

# 1. Short Context Exact Parity Check (T <= 2048)
test_ids = tok.apply_chat_template([{'role': 'user', 'content': 'Verify O(n log n) Log-Linear KV Cache parity.'}], tokenize=True, add_generation_prompt=True, return_tensors='pt')
if hasattr(test_ids, 'input_ids'):
    test_ids = test_ids.input_ids
elif isinstance(test_ids, dict):
    test_ids = test_ids['input_ids']
test_ids = test_ids.to('cuda:0')
set_loglinear_kv_runtime(enabled=False)
with torch.no_grad():
    logits_std = net.forward_logits(test_ids).detach().float()

set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)
with torch.no_grad():
    logits_loglin_short = net.forward_logits(test_ids).detach().float()

short_diff = float((logits_std - logits_loglin_short).abs().max().item())
print(f'1. Short-Context Bypass Parity (T={test_ids.shape[1]} <= 2048): max_abs_diff = {short_diff:.8f} (expected 0.0)')
assert short_diff == 0.0, f'Expected exact 0.0 diff for T <= threshold, got {short_diff}'

# 2. Long-Context Active O(n log n) Compression Test (Simulated 3,200-token context with threshold=1024)
long_seq_len = 3200
synth_long_ids = test_ids.repeat(1, (long_seq_len // test_ids.shape[1]) + 1)[:, :long_seq_len].contiguous()

set_loglinear_kv_runtime(enabled=False)
with torch.no_grad():
    from transformers.cache_utils import DynamicCache
    cache_std = DynamicCache(config=net._make_loop_cache_config())
    _ = net.forward_logits(synth_long_ids[:, :-1], past_key_values=cache_std)
    t0 = time.perf_counter()
    next_logits_std = net.forward_logits(
        synth_long_ids[:, -1:], past_key_values=cache_std, prev_ids=synth_long_ids[:, -3:-1]
    )[:, -1, :].float()
    t_std_ms = (time.perf_counter() - t0) * 1000.0

set_loglinear_kv_runtime(
    enabled=True,
    exact_threshold=1024,
    sink_tokens=32,
    local_window=512,
    base_block=32,
    hip_top_blocks=8,
)
with torch.no_grad():
    cache_ll = DynamicCache(config=net._make_loop_cache_config())
    _ = net.forward_logits(synth_long_ids[:, :-1], past_key_values=cache_ll)
    t0 = time.perf_counter()
    next_logits_ll = net.forward_logits(
        synth_long_ids[:, -1:], past_key_values=cache_ll, prev_ids=synth_long_ids[:, -3:-1]
    )[:, -1, :].float()
    t_ll_ms = (time.perf_counter() - t0) * 1000.0

cos_sim = float(F.cosine_similarity(next_logits_std, next_logits_ll, dim=-1).item())
top1_match = int(next_logits_std.argmax(dim=-1).item() == next_logits_ll.argmax(dim=-1).item())
print(f'2. Active O(n log n) NoPE Compression at T={long_seq_len}:')
print(f'   Logit Cosine Similarity vs Full O(n^2) = {cos_sim:.6f} | Top-1 Token Match = {bool(top1_match)}')
print(f'   Decode Step Latency: Full={t_std_ms:.2f} ms | Log-Linear O(log n)={t_ll_ms:.2f} ms')

# Keep standard 2048 threshold enabled for interactive use
set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)

In [ ]:
# === Cell 8: Interactive Multi-Turn Chat & Agentic <tool_call> Sandbox ===
import json, re

CHAT_HISTORY = []

def chat_turn(user_message: str, max_new_tokens: int = 262144, enable_thinking: bool = True):
    """Run an interactive multi-turn chat turn with streaming <think>...</think> output."""
    CHAT_HISTORY.append({'role': 'user', 'content': user_message})
    print(f'\nUser: {user_message}\n' + '-' * 80)
    res = generate_sparkx_loop(
        CHAT_HISTORY,
        max_new_tokens=max_new_tokens,
        enable_thinking=enable_thinking,
        temperature=0.0,
        stream=True,
    )
    CHAT_HISTORY.append({'role': 'assistant', 'content': res['answer']})
    return res

# Quick sanity turn
_ = chat_turn(
    "In 3 concise bullet points, summarize how your 60-layer Sublayer Dual-mHC recurrence and 3-Tier MoE-Engram work together during reasoning.",
    max_new_tokens=262144,
)

In [ ]:
# === Cell 7 (Optional): 8K Long-Context Post-Training Cure (`SEQ_LEN = 8192`) for `TassP0lar/Spark-X2.6-4B-Loop-mHC` ===
# Why this helps:
#   - `v0.2` (`R8 -> R16 -> R24`) was trained at `MAX_SEQ_LENGTH = 2048` and sliced long `<think>` traces at `think_ids[:max_think] + </think>`,
#     which taught the checkpoint to compress code into ~600 tokens and occasionally spike `</think>` around token ~1,000.
#   - Running this 60-step Long-Context Post-Training pass at `SEQ_LEN = 8192` with ZERO mid-thought slicing teaches the 60-layer
#     Sublayer Dual-mHC + MoE-Engram model to smoothly transition from `<think>...</think>` into full 4,000-8,000 token deliverables!
RUN_LONGCTX_POSTTRAIN = False  # Set True to run the ~12-minute 8K Long-Context Post-Training pass on GPU!
PUSH_HEALED_TO_HUB = False     # Set True to upload the 8K-healed checkpoint back to `TassP0lar/Spark-X2.6-4B-Loop-mHC`

if RUN_LONGCTX_POSTTRAIN:
    import random, time
    from datasets import load_dataset
    from sparkx_loop_core import set_trainables, build_optimizer, loop_modules_state
    from safetensors.torch import save_file

    LONG_SEQ_LEN = 8192
    POSTTRAIN_STEPS = 60
    GRAD_ACCUM = 8

    print(f'Streaming long-form un-truncated <think> + full code/reasoning samples (up to {LONG_SEQ_LEN} tokens)...')
    long_sources = [
        ('TeichAI/claude-opus-4.6-800x', 200),
        ('crownelius/Opus-4.6-Reasoning-3300x', 250),
        ('Tesslate/UIGEN-T1.5-Dataset', 200),
        ('LeafSpark/Fable-4B-Thinking-13K', 250),
    ]
    cli_leak_markers = ('claude-projects', 'c:\\users\\', 'let me check the workspace', 'pulling the full three.js')
    long_sequences = []

    for ds_id, cap in long_sources:
        kept = 0
        try:
            ds = load_dataset(ds_id, split='train', streaming=True, token=HF_TOKEN or None)
            for row in ds:
                msgs = row.get('messages') or row.get('conversations') or []
                u_txt, a_txt, th_txt = '', '', ''
                if isinstance(msgs, list) and len(msgs) >= 2:
                    for m in msgs:
                        if not isinstance(m, dict):
                            continue
                        r = str(m.get('role') or m.get('from') or '').lower()
                        c = str(m.get('content') or m.get('value') or '').strip()
                        if r in ('user', 'human') and not u_txt:
                            u_txt = c
                        elif r in ('assistant', 'gpt', 'model') and c:
                            a_txt = c
                            th_txt = str(m.get('reasoning_content') or m.get('thinking') or '').strip()
                else:
                    u_txt = str(row.get('prompt') or row.get('instruction') or row.get('question') or '').strip()
                    a_txt = str(row.get('response') or row.get('output') or row.get('answer') or '').strip()
                    th_txt = str(row.get('reasoning') or row.get('thinking') or row.get('thought') or '').strip()
                if '<think>' in a_txt and '</think>' in a_txt:
                    th_txt = a_txt.split('<think>', 1)[1].split('</think>', 1)[0].strip()
                    a_txt = a_txt.split('</think>', 1)[1].strip()
                if not u_txt or len(a_txt) < 400 or len(th_txt) < 80:
                    continue
                comb_low = (th_txt + ' ' + a_txt).lower()
                if any(mk in comb_low for mk in cli_leak_markers):
                    continue
                # Encode ONLY complete, un-truncated examples that fit inside LONG_SEQ_LEN (NEVER slice <think> mid-sentence!)
                p_str = tok.apply_chat_template(
                    [{'role': 'system', 'content': DIRECT_CHAT_SYSTEM_PROMPT}, {'role': 'user', 'content': u_txt}],
                    tokenize=False, add_generation_prompt=True, enable_thinking=True, reasoning_effort='high'
                )
                if not p_str.rstrip().endswith('<think>'):
                    p_str += '<think>\n'
                p_ids = tok.encode(p_str, add_special_tokens=False)
                t_ids = tok.encode(f'{th_txt}\n</think>\n{a_txt}', add_special_tokens=False) + [tok.eos_token_id]
                if 1024 <= (len(p_ids) + len(t_ids)) <= LONG_SEQ_LEN:
                    full_in = p_ids + t_ids
                    full_lb = ([-100] * len(p_ids)) + t_ids
                    long_sequences.append((torch.tensor(full_in, dtype=torch.long), torch.tensor(full_lb, dtype=torch.long)))
                    kept += 1
                    if kept >= cap:
                        break
            print(f'  Loaded {kept} un-truncated long-context samples from {ds_id}')
        except Exception as exc:
            print(f'  Skipped {ds_id}: {exc}')

    random.shuffle(long_sequences)
    print(f'Total un-truncated long-context sequences (1K..8K tokens): {len(long_sequences)}')

    # Train Sublayer Dual-mHC + Pass Adapters + 3-Tier MoE-Engrams + top layers on complete 8K sequences
    set_loglinear_kv_runtime(enabled=False)
    net.enable_checkpoint = True
    trainables = set_trainables(net, full_layer_ids=list(range(20, 36)), full_finetune=False)
    opt = build_optimizer(net, lr_adapter=5e-5, lr_full=5e-6, lr_mhc=1e-3, use_8bit=True)
    net.train()

    for step in range(1, POSTTRAIN_STEPS + 1):
        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        for _ in range(GRAD_ACCUM):
            inp, lbl = random.choice(long_sequences)
            xb = inp.unsqueeze(0).to('cuda:0')
            yb = lbl.unsqueeze(0).to('cuda:0')
            out = net(xb, labels=yb)
            loss = out['loss'] / GRAD_ACCUM
            loss.backward()
            step_loss += float(loss.item())
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        if step % 5 == 0 or step == 1:
            print(f'  [8K LongCtx Post-Train Step {step:3d}/{POSTTRAIN_STEPS}] loss = {step_loss:.4f} | len = {xb.shape[1]} tokens')

    net.eval()
    set_loglinear_kv_runtime(enabled=True, exact_threshold=2048)
    print('8K Long-Context Post-Training Complete! Re-run Cell 4 to test the healed model.')
